<a href="https://colab.research.google.com/github/kyeeun7706-coder/e-waste-location-optimization/blob/main/%EC%A0%84%EC%A2%85%EC%84%A4_0330.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

라이브러리 설치

In [14]:
!pip install geopandas osmnx networkx rasterio rasterstats shapely

라이브러리 임포트 + 드라이브 마운트

In [15]:
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import networkx as nx
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import Point

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


파일 경로 설정

In [16]:
BASE = '/content/drive/MyDrive/전종설'

# 지표 1 - 행정동 경계 (폴더 안 .shp)
DONG_SHP = f'{BASE}/지표1_행정구역경계/bnd_dong_11180_2025_2Q.shp'

# 지표 1 - 연령별 인구 CSV
POP_CSV = f'{BASE}/지표1_202412_202512_금천구_연령별인구현황_연간.csv'

# 지표 2 - DEM 래스터
DEM_IMG = f'{BASE}/지표2_서울dem.img'

# 지표 2 - 격자별 고령인구 (폴더 안 .shp)
GRID_OLD_SHP = f'{BASE}/지표2_금천구격자별고령인구수/vl_blk.shp'

# 지표 2 - 격자별 총인구 (폴더 안 .shp)
GRID_TOT_SHP = f'{BASE}/지표2_금천구격자별총인구수/vl_blk.shp'

# 지표 3 - 수거함 8개 위치
BINS_CSV = f'{BASE}/지표3_금천구폐가전수거함8개위치.csv'

파일 정상 로드 확인

In [17]:
# 행정동 경계
dong = gpd.read_file(DONG_SHP)
print("행정동 경계 컬럼:", dong.columns.tolist())
print(dong.head(3))

행정동 경계 컬럼: ['BASE_DATE', 'ADM_CD', 'ADM_NM', 'geometry']
  BASE_DATE    ADM_CD ADM_NM  \
0  20250630  11180510    가산동   
1  20250630  11180520   독산1동   
2  20250630  11180530   독산2동   

                                            geometry  
0  POLYGON ((945134.075 1943182.19, 945138.213 19...  
1  POLYGON ((946851.545 1942312.478, 946849.923 1...  
2  POLYGON ((947770.689 1941031.612, 947770.422 1...  


In [19]:
# 연령별 인구 CSV
pop = pd.read_csv(POP_CSV, encoding='EUC-KR')
print("인구 CSV 컬럼:", pop.columns.tolist())
print(pop.head(3))

인구 CSV 컬럼: ['행정구역', '2024년_거주자_총인구수', '2024년_거주자_연령구간인구수', '2024년_거주자_65~69세', '2024년_거주자_70~74세', '2024년_거주자_75~79세', '2024년_거주자_80~84세', '2024년_거주자_85~89세', '2024년_거주자_90~94세', '2024년_거주자_95~99세', '2024년_거주자_100세 이상', '2024년_남_거주자_총인구수', '2024년_남_거주자_연령구간인구수', '2024년_남_거주자_65~69세', '2024년_남_거주자_70~74세', '2024년_남_거주자_75~79세', '2024년_남_거주자_80~84세', '2024년_남_거주자_85~89세', '2024년_남_거주자_90~94세', '2024년_남_거주자_95~99세', '2024년_남_거주자_100세 이상', '2024년_여_거주자_총인구수', '2024년_여_거주자_연령구간인구수', '2024년_여_거주자_65~69세', '2024년_여_거주자_70~74세', '2024년_여_거주자_75~79세', '2024년_여_거주자_80~84세', '2024년_여_거주자_85~89세', '2024년_여_거주자_90~94세', '2024년_여_거주자_95~99세', '2024년_여_거주자_100세 이상', '2025년_거주자_총인구수', '2025년_거주자_연령구간인구수', '2025년_거주자_65~69세', '2025년_거주자_70~74세', '2025년_거주자_75~79세', '2025년_거주자_80~84세', '2025년_거주자_85~89세', '2025년_거주자_90~94세', '2025년_거주자_95~99세', '2025년_거주자_100세 이상', '2025년_남_거주자_총인구수', '2025년_남_거주자_연령구간인구수', '2025년_남_거주자_65~69세', '2025년_남_거주자_70~74세', '2025년_남_거주자_75~79세', '2025년_남_거주자_80~84세', '2025년_남_

In [20]:
# 격자별 고령인구
grid_old = gpd.read_file(GRID_OLD_SHP)
print("격자 고령인구 컬럼:", grid_old.columns.tolist())
print(grid_old.head(3))

격자 고령인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [21]:
# 격자별 총인구
grid_tot = gpd.read_file(GRID_TOT_SHP)
print("격자 총인구 컬럼:", grid_tot.columns.tolist())
print(grid_tot.head(3))

격자 총인구 컬럼: ['gid', 'lbl', 'val', 'geometry']
            gid   lbl  val                                           geometry
0  ë¤ì¬444429  None  NaN  POLYGON ((944400 1942900, 944400 1943000, 9445...
1  ë¤ì¬444430  None  NaN  POLYGON ((944400 1943000, 944400 1943100, 9445...
2  ë¤ì¬444431  None  NaN  POLYGON ((944400 1943100, 944400 1943200, 9445...


In [22]:
# DEM
with rasterio.open(DEM_IMG) as src:
    print("DEM CRS:", src.crs)
    print("DEM 해상도:", src.res)
    print("DEM 크기:", src.width, "x", src.height)

DEM CRS: EPSG:5179
DEM 해상도: (90.0, 90.0)
DEM 크기: 253 x 316


In [23]:
# 수거함 위치
bins_df = pd.read_csv(BINS_CSV)
print("수거함 컬럼:", bins_df.columns.tolist())
print(bins_df)

수거함 컬럼: ['id', 'name', 'lat', 'lon']
   id   name        lat         lon
0   1  수거함_1  37.469710  126.898966
1   2  수거함_2  37.465816  126.897190
2   3  수거함_3  37.462436  126.898027
3   4  수거함_4  37.463940  126.891584
4   5  수거함_5  37.462466  126.890374
5   6  수거함_6  37.457927  126.897581
6   7  수거함_7  37.460229  126.886538
7   8  수거함_8  37.451069  126.897925
